In [ ]:
#Import the necessary libraries
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern
from sklearn.preprocessing import StandardScaler
from scipy.stats import qmc
import numpy as np
from scipy.optimize import minimize

In [ ]:
X = np.array([
    [0.19144708, 0.03819337, 0.60741781, 0.41458414],
    [0.75865295, 0.53651774, 0.65600038, 0.36034155],
    [0.43834987, 0.8043397, 0.21024527, 0.15129482],
    [0.70605083, 0.53419196, 0.26424335, 0.48208755],
    [0.83647799, 0.19360965, 0.6638927, 0.78564888],
    [0.68343225, 0.11866264, 0.82904591, 0.56757661],
    [0.55362148, 0.66734998, 0.32380582, 0.81486975],
    [0.35235627, 0.32224153, 0.11697937, 0.47311252],
    [0.15378571, 0.72938169, 0.42259844, 0.44307417],
    [0.46344227, 0.63002451, 0.10790646, 0.9576439],
    [0.67749115, 0.35850951, 0.47959222, 0.07288048],
    [0.58397341, 0.14724265, 0.34809746, 0.42861465],
    [0.30688872, 0.31687813, 0.62263448, 0.09539906],
    [0.51114177, 0.817957, 0.72871042, 0.11235362],
    [0.43893338, 0.77409176, 0.37816709, 0.93369621],
    [0.22418902, 0.84648049, 0.87948418, 0.87851568],
    [0.72526172, 0.47987049, 0.08894684, 0.75976022],
    [0.35548161, 0.63961937, 0.41761768, 0.12260384],
    [0.11987923, 0.86254031, 0.64333133, 0.84980383],
    [0.12688467, 0.15342962, 0.77016219, 0.19051811],
    [0.224189, 0.846480, 0.879484, 0.878515],
    [0.863052, 0.425156, 0.725479, 0.801889],
    [0.22418902, 0.84648049, 0.87948418, 0.87851568],
    [0.385144, 0.835222, 0.875492, 0.956764],
    [0.267906, 0.465524, 0.221346, 0.494072],
    [0.609217, 0.044599, 0.293169, 0.514960],
    [0.735676, 0.105094, 0.849976, 0.954765],
    [0.573733, 0.737421, 0.878145, 0.950371],
    [0.448753, 0.222797, 0.878959, 0.949337],
    [0.467826, 0.854649, 0.804835, 0.941106],
    [0.149163, 0.704136, 0.873139, 0.950363],
    [0.347797, 0.833399, 0.861106, 0.921566]
])

y = np.array([
    6.44434399e+01, 1.83013796e+01, 1.12939795e-01, 4.21089813e+00,
    2.58370525e+02, 7.84343889e+01, 5.75715369e+01, 1.09571876e+02,
    8.84799176e+00, 2.33223610e+02, 2.44230883e+01, 6.44201468e+01,
    6.34767158e+01, 7.97291299e+01, 3.55806818e+02, 1.08885962e+03,
    2.88667516e+01, 4.51815703e+01, 4.31612757e+02, 9.97233189e+00,
    1088.85351147375, 472.699581504888, 1088.85351147375, 1513.30490292936,
    88.61613670263588, 44.56790661878319, 976.8066045249683, 1285.4697786864638,
    677.8614123107656, 1228.9334442204602, 971.9245547045626, 1194.0392012268128
])

#shape of X & y
print(X.shape)
print(y.shape)

In [ ]:
# GP setup
kernel = ConstantKernel(1.0) * Matern(
    nu=2.5,
    length_scale=[1.0, 1.0, 1.0, 1.0],
    length_scale_bounds=(1e-3, 1e4)
)

gpr = GaussianProcessRegressor(
        kernel=kernel,
        n_restarts_optimizer=20,
        alpha=1e-6,
        normalize_y=True
    )

# Fit on your 4D data
gpr.fit(X, y)


# Check what the GP learned
print("GP Model Diagnostics:")
print(f"  Kernel: {gpr.kernel_}")
print(f"  Length scales: {gpr.kernel_.k2.length_scale}")
print(f"  Training score: {gpr.score(X, y):.3f}") # Reshape y for scoring

# Define bounds for 3D space
bounds = [
    (X[:, 0].min(), X[:, 0].max()),  # x1 bounds
    (X[:, 1].min(), X[:, 1].max()),  # x2 bounds
    (X[:, 2].min(), X[:, 2].max()),  # x3 bounds
    (X[:, 3].min(), X[:, 3].max())   # x4 bounds
]
print(f"  Bounds: {bounds}")

# Diagnostics
print(f"Length scales: {gpr.kernel_.k2.length_scale}")
print(f"Training R²: {gpr.score(X, y):.3f}")

# Generate 3D candidates
sampler = qmc.LatinHypercube(d=4)
X_candidates = qmc.scale(
    sampler.random(n=20000),
    l_bounds=[b[0] for b in bounds],
    u_bounds=[b[1] for b in bounds]
)

# Predict on candidates
y_pred, y_std = gpr.predict(X_candidates, return_std=True)

# UCB acquisition function
kappa = 2.0
ucb = y_pred + kappa * y_std

# Find best point
best_idx = np.argmax(ucb)
x_next = X_candidates[best_idx]

print(f"\nNext Point to Sample:")
print(f"  X = {x_next}")  # Should show 3 values
print(f"  Predicted y = {y_pred[best_idx]:.4f}")
print(f"  Uncertainty = {y_std[best_idx]:.4f}")
print(f"  UCB score = {ucb[best_idx]:.4f}")

# Show top 5 candidates
top5_idx = np.argsort(ucb)[-5:][::-1]
print(f"\nTop 5 Candidates:")
for i, idx in enumerate(top5_idx, 1):
    print(f"  {i}. X={X_candidates[idx]}, "
          f"pred={y_pred[idx]:.3f}, std={y_std[idx]:.3f}, ucb={ucb[idx]:.3f}")
    # add X candidate to X
    #X = np.vstack((X, X_candidates[idx]))
    # Reshape y to (N, 1) if it's not already, and reshape the new prediction to (1, 1)
    # It's better to ensure y1 is always (N, 1) from the start, but for an immediate fix:
    #if y.ndim == 1:
     #   y = y.reshape(-1, 1)
   # y = np.vstack((y, y_pred[idx].reshape(-1, 1)))

In [ ]:
#shape of X & y
print(X.shape)
print(y.shape)

Trying PyTorch & TensorFlow or this week

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from scipy.stats import qmc

In [ ]:
# STEP 1: Define PyTorch Model
# ========================================
class NNSurrogate(nn.Module):
    def __init__(self, input_dim, hidden_sizes=[64, 32], dropout=0.2):
        super(NNSurrogate, self).__init__()

        layers = []
        prev_size = input_dim

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size

        layers.append(nn.Linear(prev_size, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [ ]:
#========================================
# STEP 2: Train on Your Data
# ========================================
def train_on_your_data(X, y, epochs=1000, lr=0.001):
    """
    Train PyTorch model on your X and y
    """
    n_samples, input_dim = X.shape
    print(f"Training on {n_samples} samples, {input_dim}D")

    # Convert to tensors
    X_tensor = torch.FloatTensor(X)
    y_tensor = torch.FloatTensor(y).reshape(-1, 1)

    # Normalize
    X_mean, X_std = X_tensor.mean(0), X_tensor.std(0) + 1e-8
    y_mean, y_std = y_tensor.mean(), y_tensor.std() + 1e-8

    X_norm = (X_tensor - X_mean) / X_std
    y_norm = (y_tensor - y_mean) / y_std

    # Create model
    model = NNSurrogate(input_dim, hidden_sizes=[64, 32], dropout=0.2)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.01)

    # Train
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        predictions = model(X_norm)
        loss = criterion(predictions, y_norm)
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.6f}")

    # Evaluate on training data
    model.eval()
    with torch.no_grad():
        train_pred = model(X_norm)
        train_pred_denorm = train_pred * y_std + y_mean
        mse = ((train_pred_denorm - y_tensor) ** 2).mean().item()
        r2 = 1 - mse / y_tensor.var().item()

    print(f"\nTraining Results:")
    print(f"  MSE: {mse:.6f}")
    print(f"  R²: {r2:.4f}")

    return model, X_mean, X_std, y_mean, y_std

In [ ]:
# Train the model
model, X_mean, X_std, y_mean, y_std = train_on_your_data(X, y, epochs=1000)

In [ ]:
# ========================================
# STEP 3: Predict on New Points
# ========================================
def predict_new_points(model, X_new, X_mean, X_std, y_mean, y_std):
    """
    Predict on new X points
    """
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_new)
        X_norm = (X_tensor - X_mean) / X_std
        y_pred_norm = model(X_norm)
        y_pred = y_pred_norm * y_std + y_mean

    return y_pred.numpy().flatten()

# generating new candidates

n_dims = X.shape[1]

# Define bounds for 2D space
bounds = [
    (X[:, i].min(), X[:, i].max()) for i in range(n_dims)
]

# Generate candidates
sampler = qmc.LatinHypercube(d=n_dims)
X_candidates = qmc.scale(
    sampler.random(n=10000),
    l_bounds=[b[0] for b in bounds],
    u_bounds=[b[1] for b in bounds]
 )


y_pred = predict_new_points(model, X_candidates, X_mean, X_std, y_mean, y_std)
print(f"\nPredictions on new points:")
print(f"X_new: {X_candidates}")
print(f"y_pred: {y_pred}")

#print first X candidate and y pred value
print(f"First X candidate: {X_candidates[0]}")
print(f"First y prediction: {y_pred[0]}")

In [ ]:
class BayesianOptimizer:
    def __init__(self, bounds, n_initial=10, kernel=None, alpha=0.01, n_restarts=30):
        """
        Bayesian Optimization for hyperparameter tuning

        Args:
            bounds: List of (min, max) tuples for each dimension
            n_initial: Number of random initial points
            kernel: GP kernel (defaults to Matern)
            alpha: Noise regularization
            n_restarts: GP optimizer restarts
        """
        self.bounds = np.array(bounds)
        self.dim = len(bounds)
        self.n_initial = n_initial
        self.alpha = alpha
        self.n_restarts = n_restarts

        # Initialize scalers
        self.scaler_X = StandardScaler()
        self.scaler_y = StandardScaler()

        # Setup kernel
        if kernel is None:
            self.kernel = ConstantKernel(1.0, constant_value_bounds=(1e-2, 1e3)) * \
                         Matern(nu=2.5,
                                length_scale=np.ones(self.dim),
                                length_scale_bounds=(0.01, 1000.0))
        else:
            self.kernel = kernel

        # Initialize GP
        self.gpr = GaussianProcessRegressor(
            kernel=self.kernel,
            n_restarts_optimizer=self.n_restarts,
            alpha=self.alpha,
            normalize_y=False,
            random_state=42
        )

        # Storage for observations - Initialize as lists and populate with initial data
        self.X_observed = X.tolist()  # Convert initial numpy array to list
        self.y_observed = y.tolist()  # Convert initial numpy array to list

        # Convert observed lists to numpy arrays for initial scaling and GP fitting
        X_initial_np = np.array(self.X_observed)
        y_initial_np = np.array(self.y_observed).reshape(-1, 1)

        # Fit scalers on the initial data provided
        self.scaler_X.fit(X_initial_np)
        self.scaler_y.fit(y_initial_np)

        # Fit GP on initial scaled data
        X_scaled_initial = self.scaler_X.transform(X_initial_np)
        y_scaled_initial = self.scaler_y.transform(y_initial_np).ravel()
        self.gpr.fit(X_scaled_initial, y_scaled_initial)

        # Print initial diagnostics
        r2 = self.gpr.score(X_scaled_initial, y_scaled_initial)
        print(f"Initial GP Model Fitted with {len(self.y_observed)} observations:")
        print(f"  Training R²: {r2:.3f}")
        if hasattr(self.gpr, 'kernel_'):
            print(f"  Length scales: {self.gpr.kernel_.k2.length_scale}")
        if r2 > 0.99:
            print("  ⚠️  WARNING: Potential overfitting (R²>0.99) on initial data")

    def _get_initial_points(self):
        """Generate initial points using Latin Hypercube Sampling"""
        sampler = qmc.LatinHypercube(d=self.dim, seed=42)
        points = qmc.scale(
            sampler.random(n=self.n_initial),
            l_bounds=self.bounds[:, 0],
            u_bounds=self.bounds[:, 1]
        )
        return points

    def _acquisition_ucb(self, X, kappa=2.0):
        """Upper Confidence Bound acquisition function"""
        # Ensure scalers are fitted before transforming
        # This check is now redundant due to fitting in __init__, but kept for robustness
        if len(self.X_observed) == 0:
            raise RuntimeError("Scaler not fitted. Call update() with initial data first.")

        X_scaled = self.scaler_X.transform(X.reshape(1, -1))
        y_pred_scaled, y_std_scaled = self.gpr.predict(X_scaled, return_std=True)

        # Unscale
        y_pred = self.scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
        y_std = y_std_scaled * self.scaler_y.scale_[0]

        # UCB (negate for minimization)
        return -(y_pred + kappa * y_std)[0]

    def _acquisition_ei(self, X, xi=0.01):
        """Expected Improvement acquisition function"""
        from scipy.stats import norm

        # Ensure scalers are fitted before transforming
        # This check is now redundant due to fitting in __init__, but kept for robustness
        if len(self.X_observed) == 0:
            raise RuntimeError("Scaler not fitted. Call update() with initial data first.")

        X_scaled = self.scaler_X.transform(X.reshape(1, -1))
        y_pred_scaled, y_std_scaled = self.gpr.predict(X_scaled, return_std=True)

        # Unscale
        y_pred = self.scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()[0]
        y_std = y_std_scaled[0] * self.scaler_y.scale_[0]

        # Current best
        y_best = np.max(self.y_observed)

        # Avoid division by zero
        if y_std < 1e-10:
            return 0.0

        # Calculate EI
        z = (y_pred - y_best - xi) / y_std
        ei = (y_pred - y_best - xi) * norm.cdf(z) + y_std * norm.pdf(z)

        return -ei  # Negate for minimization

    def _propose_location(self, acquisition='ucb', kappa=2.0, xi=0.01, n_restarts=25):
        """Propose next sampling point by optimizing acquisition function"""

        # Choose acquisition function
        if acquisition == 'ucb':
            acq_func = lambda x: self._acquisition_ucb(x, kappa=kappa)
        elif acquisition == 'ei':
            acq_func = lambda x: self._acquisition_ei(x, xi=xi)
        else:
            raise ValueError(f"Unknown acquisition function: {acquisition}")

        # Multi-start optimization
        min_val = float('inf')
        min_x = None

        # Generate random starting points
        sampler = qmc.LatinHypercube(d=self.dim, seed=None)
        x0_samples = qmc.scale(
            sampler.random(n=n_restarts), # n_restarts for optimization starts
            l_bounds=self.bounds[:, 0],
            u_bounds=self.bounds[:, 1]
        )

        for x0 in x0_samples:
            result = minimize(
                acq_func,
                x0=x0,
                bounds=self.bounds,
                method='L-BFGS-B'
            )

            if result.fun < min_val:
                min_val = result.fun
                min_x = result.x

        return min_x

    def update(self, X_new, y_new):
        """Add new observations and refit GP"""
        # Ensure X_new is always treated as a 1D array for single point or a list of 1D arrays for multiple points
        if isinstance(X_new, np.ndarray) and X_new.ndim == 1:
            self.X_observed.append(X_new.tolist()) # Convert to list before appending to self.X_observed list
        elif isinstance(X_new, list) and all(isinstance(x, (list, np.ndarray)) for x in X_new):
            # If X_new is a list of lists/arrays, or a 2D numpy array
            for x_val in X_new:
                self.X_observed.append(x_val.tolist() if isinstance(x_val, np.ndarray) else x_val)
        else:
            # Assuming it's a single point that might not be a numpy array or list itself
            self.X_observed.append(X_new)

        if np.isscalar(y_new):
            self.y_observed.append(y_new)
        elif isinstance(y_new, (list, np.ndarray)):
            self.y_observed.extend(y_new)
        else:
            self.y_observed.append(y_new)

        # Convert observed lists to numpy arrays for scaling and GP fitting
        X_current = np.array(self.X_observed)
        y_current = np.array(self.y_observed).reshape(-1, 1)

        # Scale data - fit_transform on the growing dataset
        X_scaled = self.scaler_X.fit_transform(X_current)
        y_scaled = self.scaler_y.fit_transform(y_current).ravel()

        # Fit GP
        self.gpr.fit(X_scaled, y_scaled)

        # Print diagnostics
        r2 = self.gpr.score(X_scaled, y_scaled)
        print(f"\nGP Model Updated:")
        print(f"  Observations: {len(self.y_observed)}")
        print(f"  Training R²: {r2:.3f}")
        if hasattr(self.gpr, 'kernel_'):
            print(f"  Length scales: {self.gpr.kernel_.k2.length_scale}")
        if r2 > 0.99:
            print("  ⚠️  WARNING: Potential overfitting (R²>0.99)")

    def suggest(self, acquisition='ucb', kappa=2.0, xi=0.01):
        """Suggest next point to evaluate"""
        if len(self.y_observed) < self.n_initial:
            # Use random exploration initially
            # Ensure a distinct initial point is returned each time
            if len(self.X_observed) < self.n_initial:
                new_random_points = self._get_initial_points()
                # Find a point not already in self.X_observed (or similar logic)
                # For simplicity here, just return the next sequential initial point
                return new_random_points[len(self.X_observed)]
            else:
                # If we have enough observed points but not yet hit n_initial for proposal
                # This case is tricky if n_initial is smaller than initial X,y provided
                # Given the fix in __init__ this block will likely not be hit for new random points unless X_observed is explicitly cleared
                return self._propose_location(acquisition=acquisition, kappa=kappa, xi=xi)
        else:
            # Use acquisition function
            return self._propose_location(acquisition=acquisition, kappa=kappa, xi=xi)

    def get_best(self):
        """Return best observed point"""
        if len(self.y_observed) == 0:
            return None, None

        best_idx = np.argmax(self.y_observed)
        return np.array(self.X_observed[best_idx]), self.y_observed[best_idx]


In [ ]:
# ==========================
# USAGE EXAMPLE
# ==========================

# Define your 5D bounds
bounds = [
    (0.0, 1.0),   # Parameter 1
    (0.0, 1.0),   # Parameter 2
    (0.0, 1.0),   # Parameter 3
    (0.0, 1.0),   # Parameter 4
]

# Initialize optimizer
optimizer = BayesianOptimizer(
    bounds=bounds,
    n_initial=20,      # Initial random samples
    alpha=0.01,        # GP noise
    n_restarts=30      # GP hyperparameter optimization restarts
)

# Define your objective function
def objective_function(params):
    """
    Your black-box function to optimize

    Args:
        params: Array of 4 parameters
    Returns:
        score: Float (higher is better)
    """
    # Example: replace with your actual function
    # e.g., train model with these hyperparameters and return validation score
    score = -np.sum((params - 0.5)**2)  # Dummy function
    return score

# Bayesian Optimization Loop
n_iterations = 50

print("Starting Bayesian Optimization...\n")

for i in range(n_iterations):
    # Get suggestion
    x_next = optimizer.suggest(acquisition='ucb', kappa=2.0)

    # Evaluate objective
    y_next = objective_function(x_next)

    # Update optimizer
    optimizer.update(x_next, y_next)

    # Get current best
    x_best, y_best = optimizer.get_best()

    print(f"Iteration {i+1}/{n_iterations}")
    print(f"  Suggested: {x_next}")
    print(f"  Score: {y_next:.4f}")
    print(f"  Best so far: {y_best:.4f}")
    print(f"  Best params: {x_best}\n")

# Final result
x_best, y_best = optimizer.get_best()
print("\n" + "="*50)
print("OPTIMIZATION COMPLETE")
print("="*50)
print(f"Best score: {y_best:.4f}")
print(f"Best parameters: {x_best}")